# Hybrid Resolver - lexical + RAG integration

`%run rag_resolver.ipynb` (which `%run`s the lexical `resolver.ipynb` and adds the RAG
primitives), then wires them together: `merge_semantic` + `resolve_meeting_context_hybrid`
(lexical first, semantic memory as a low-confidence fallback), plus the eval/threshold
tuning for the hybrid path. This is the integration + test file; `rag_resolver.ipynb`
stays pure RAG primitives and `pipeline.ipynb` stays pure orchestration.

In [1]:
import os
# Run from backend/ so `%run` resolves regardless of the kernel's launch dir.
if not os.path.exists("common.ipynb"):
    for _p in ("backend", "extras/socialApp/backend"):
        if os.path.exists(f"{_p}/common.ipynb"):
            os.chdir(_p)
            break
_HYB_DEMO_TOP = globals().get("RUN_DEMO", True)
_HYB_LIVE_TOP = globals().get("RUN_LIVE", False)
RUN_DEMO = RUN_LIVE = False   # load the RAG + lexical base quietly
%run rag_resolver.ipynb
RUN_DEMO, RUN_LIVE = _HYB_DEMO_TOP, _HYB_LIVE_TOP

common ready. cwd=backend TODAY=2026-06-16
TalkingPoint, TopicTheme, PlannerOutput, BrainstormOutput, render_brainstorm defined.
Person, QueryClues, Candidate, ResolutionResult, Interaction, Fact, PrepSession defined.
store ready: u1 has 6 people, 1 interactions.
name matching (damerau_levenshtein, soundex, name_match_score) defined.
generate_candidates defined.
decide + resolve_from_clues defined.
clues_agent + extract_clues defined.
resolve_meeting_context + show_resolution defined.
RouteDecision + route_resolution defined.
select_candidate defined.
write-back defined: save_prep_session / get_prep_sessions / propose_new_person / save_new_person / find_merge_candidates / apply_merge / propose_fact / add_fact
intent_agent + classify_intent defined.
recall (render_recall_answer/apply_recall/run_recall) + update (extract_update/apply_update/run_update) defined.
MemoryChunk + RagHit + build_memory_chunks defined.
RAG layer ready (provider=openai): embed_text / embed_chunks / cosine_simila

In [2]:
# RAG Step 3 - hybrid resolution: lexical first, semantic memory as a LOW-CONFIDENCE
# fallback. Lexical/name evidence wins on identity; RAG only fills not_found / clarify.
SEMANTIC_RESOLVE_MIN = 0.30   # min cosine for a semantic hit to count (pin by tests)
SEMANTIC_MARGIN      = 0.05   # the top person must beat the 2nd by this (cosine)


def _semantic_candidate(user_id: str, pid: str, score: float, text: str) -> Candidate:
    p = get_person(user_id, pid)
    return Candidate(target_id=pid, display_name=p.name if p else pid, score=round(score, 3),
                     name_score=0.0, evidence=[f"semantic memory match: {text[:60]}"])


def merge_semantic(lex: ResolutionResult, hits: list[RagHit], user_id: str) -> ResolutionResult:
    """Lexical confidence wins: a lexical 'resolved' or 'ambiguous' is never overridden by
    semantic. Only 'not_found' / 'clarify' fall through to semantic-memory evidence."""
    if lex.status in ("resolved", "ambiguous"):
        return lex
    best: dict[str, tuple[float, str]] = {}
    for h in hits:
        if h.person_id and (h.person_id not in best or h.score > best[h.person_id][0]):
            best[h.person_id] = (h.score, h.text)
    ranked = sorted(((pid, s, t) for pid, (s, t) in best.items()), key=lambda x: x[1], reverse=True)
    ranked = [r for r in ranked if r[1] >= SEMANTIC_RESOLVE_MIN]
    if not ranked:
        return lex
    margin = ranked[0][1] - (ranked[1][1] if len(ranked) > 1 else 0.0)
    if len(ranked) == 1 or margin >= SEMANTIC_MARGIN:
        c = _semantic_candidate(user_id, *ranked[0])
        return ResolutionResult(status="resolved", selected_targets=[c], candidates=[c],
                                extracted_clues=lex.extracted_clues,
                                resolution_summary=f"Resolved to {c.display_name} via semantic memory.")
    cands = [_semantic_candidate(user_id, pid, s, t) for pid, s, t in ranked]
    return ResolutionResult(status="ambiguous", candidates=cands, extracted_clues=lex.extracted_clues,
                            resolution_summary="Multiple semantic memory matches; ask the user.")


async def resolve_meeting_context_hybrid(
    user_id: str, utterance: str,
    use_rag: Literal["auto", "always", "never"] = "auto",
    now: datetime.date | None = None,
) -> ResolutionResult:
    """Lexical resolve first; 'auto' runs semantic only when lexical is low-confidence
    (not_found / clarify). 'always' forces the semantic call; 'never' is lexical-only.
    merge_semantic guarantees lexical resolved/ambiguous is never overridden."""
    clues = await extract_clues(utterance, now)
    lex = resolve_from_clues(clues, get_people(user_id), get_interactions(user_id))
    run_rag = (use_rag != "never" and RAG_ENABLED
               and (use_rag == "always" or lex.status in ("not_found", "clarify")))
    if not run_rag:
        return lex
    return merge_semantic(lex, await semantic_search(user_id, utterance, k=8), user_id)


print("merge_semantic + resolve_meeting_context_hybrid defined.")

merge_semantic + resolve_meeting_context_hybrid defined.


In [3]:
# Step 5 - RAG-enhanced recall: resolve via the hybrid path, then attach grounded
# memory snippets from retrieval. Retrieval supplies evidence; the answer is still
# rendered from records (the LLM never invents recall content).
def _attach_memory(result: RecallResult, hits: list[RagHit]) -> RecallResult:
    if result.action != "answer":
        return result
    # exclude the person's own profile chunk (the answer already shows the profile)
    snippets = [h.text for h in hits if h.person_id == result.person_id
                and h.metadata.get("source") != "person"][:2]
    if not snippets:
        return result
    return RecallResult(action="answer", person_id=result.person_id,
                        message=result.message + "\nRelevant memory: " + " | ".join(snippets))


async def run_recall_hybrid(user_id: str, utterance: str) -> RecallResult:
    """Recall using the hybrid resolver (so vague references resolve), enriched with
    retrieved memory snippets for the resolved person."""
    resolution = await resolve_meeting_context_hybrid(user_id, utterance)
    result = apply_recall(resolution, user_id)
    if result.action == "answer" and RAG_ENABLED:
        result = _attach_memory(result, await semantic_search(user_id, utterance, k=3))
    return result


print("run_recall_hybrid + _attach_memory defined.")

run_recall_hybrid + _attach_memory defined.


In [4]:
# Demo - offline checks for the hybrid merge logic (lexical wins; semantic fills not_found).
RUN_DEMO = globals().get("RUN_DEMO", True)
if RUN_DEMO:
    fails = 0

    def _check(label, got, want):
        global fails
        ok = got == want
        fails += 0 if ok else 1
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {got!r} (want {want!r})")

    print("Hybrid merge (lexical wins; semantic fills not_found):")
    PEOPLE["u_s3"] = [Person(id="p_x", name="Xavier"), Person(id="p_y", name="Yara")]
    _q = QueryClues()
    _cx = Candidate(target_id="p_x", display_name="Xavier", score=1.0, name_score=1.0)
    _cy = Candidate(target_id="p_y", display_name="Yara", score=1.0, name_score=1.0)
    lex_res = ResolutionResult(status="resolved", selected_targets=[_cx], candidates=[_cx], extracted_clues=_q)
    lex_amb = ResolutionResult(status="ambiguous", candidates=[_cx, _cy], extracted_clues=_q)
    lex_nf = ResolutionResult(status="not_found", extracted_clues=_q)
    h_strong = [RagHit(chunk_id="c1", target_id="p_x", person_id="p_x", score=0.60, text="loves chess")]
    h_two = [RagHit(chunk_id="c1", target_id="p_x", person_id="p_x", score=0.55, text="chess"),
             RagHit(chunk_id="c2", target_id="p_y", person_id="p_y", score=0.52, text="hiking")]
    h_weak = [RagHit(chunk_id="c1", target_id="p_x", person_id="p_x", score=0.10, text="chess")]
    _check("H1 lexical resolved not overridden", merge_semantic(lex_res, h_strong, "u_s3").status, "resolved")
    _check("H2 lexical ambiguous preserved", merge_semantic(lex_amb, h_strong, "u_s3").status, "ambiguous")
    _check("H3 not_found + strong -> resolved", merge_semantic(lex_nf, h_strong, "u_s3").status, "resolved")
    _check("H3b resolved to Xavier", merge_semantic(lex_nf, h_strong, "u_s3").selected_targets[0].target_id, "p_x")
    _check("H4 not_found + two close -> ambiguous", merge_semantic(lex_nf, h_two, "u_s3").status, "ambiguous")
    _check("H5 not_found + weak -> not_found", merge_semantic(lex_nf, h_weak, "u_s3").status, "not_found")

    print("Step 5 - recall memory enrichment (offline):")
    _ans = RecallResult(action="answer", person_id="p_jordan", message="Jordan")
    _mh = [RagHit(chunk_id="c1", target_id="i1", person_id="p_jordan", score=0.70, text="Talked about ML grad programs", metadata={"source": "interaction"}),
           RagHit(chunk_id="c0", target_id="p_jordan", person_id="p_jordan", score=0.80, text="Jordan profile blob", metadata={"source": "person"}),
           RagHit(chunk_id="c2", target_id="p2", person_id="p_other", score=0.60, text="unrelated note", metadata={"source": "interaction"})]
    _enriched = _attach_memory(_ans, _mh)
    _check("H6 interaction snippet attached", "Relevant memory" in _enriched.message and "ML grad" in _enriched.message, True)
    _check("H6b own profile chunk excluded", "profile blob" not in _enriched.message, True)
    _check("H6c other-person snippet excluded", "unrelated" not in _enriched.message, True)
    _check("H7 non-answer passes through", _attach_memory(RecallResult(action="not_found", message="none"), _mh).message, "none")

    print(f"\n{'ALL PASS' if fails == 0 else str(fails) + ' FAILURE(S)'}")

Hybrid merge (lexical wins; semantic fills not_found):
  [PASS] H1 lexical resolved not overridden: 'resolved' (want 'resolved')
  [PASS] H2 lexical ambiguous preserved: 'ambiguous' (want 'ambiguous')
  [PASS] H3 not_found + strong -> resolved: 'resolved' (want 'resolved')
  [PASS] H3b resolved to Xavier: 'p_x' (want 'p_x')
  [PASS] H4 not_found + two close -> ambiguous: 'ambiguous' (want 'ambiguous')
  [PASS] H5 not_found + weak -> not_found: 'not_found' (want 'not_found')
Step 5 - recall memory enrichment (offline):
  [PASS] H6 interaction snippet attached: True (want True)
  [PASS] H6b own profile chunk excluded: True (want True)
  [PASS] H6c other-person snippet excluded: True (want True)
  [PASS] H7 non-answer passes through: 'none' (want 'none')

ALL PASS


In [5]:
# Live demo - hybrid resolution end-to-end (real embeddings + extract_clues). OFF by default.
# Doc eval: the AI-study query is EXPECTED ambiguous (Jordan + Alex Li both plausible).
RUN_LIVE = globals().get("RUN_LIVE", False)
if RUN_LIVE and RAG_ENABLED:
    print("LIVE - hybrid resolution (auto: RAG only on low-confidence):")
    for utt in [
        "I am meeting Alex tomorrow",                                      # lexical ambiguous (RAG skipped)
        "I am meeting Alex from my CS449 group",                           # lexical resolved -> Li
        "meeting the friend who bakes bread",                              # lexical miss -> semantic -> Sarah
        "the person aiming for advanced study in artificial intelligence",  # competing semantic -> ambiguous
    ]:
        r = await resolve_meeting_context_hybrid("u1", utt)
        who = [c.display_name for c in r.selected_targets] or [c.display_name for c in r.candidates]
        print(f"  {r.status:10} {who}  <- {utt}")
        if r.selected_targets and r.selected_targets[0].evidence:
            print(f"             {r.selected_targets[0].evidence}")

    print("LIVE Step 5 - RAG-enhanced recall:")
    for q in ["tell me about Sarah", "who is the friend that bakes bread"]:
        rr = await run_recall_hybrid("u1", q)
        print(f"  [{rr.action}] {q}")
        print("    " + rr.message.replace(chr(10), " | "))
elif RUN_LIVE:
    print("RUN_LIVE set but RAG disabled (no OPENAI_API_KEY).")